# Experiment 4 — GE-MolSG vs MMFF94 (DUDE-Z retrieval)

Per-target retrieval performance (EF1%, BEDROC) for two methods:

- **GE-MolSG** — WKS → hard k-NN BoF (`knn_histogram`) over the geo codebook,
  chi-squared-kernel retrieval. Reads surface data from the DUDE-Z `ESP_Npy`
  directories.
- **MMFF94** — the same GE-MolSG pipeline applied to the precomputed
  `ESP_Npy_MMFF94` surfaces (MMFF94 partial-charge ESP), chi-squared-kernel
  retrieval.

Same workflow as Experiments 1–3: iterative per-target generation with on-disk
caching, summary, two scatter plots (BEDROC, EF1%), and a Wilcoxon test focused
on GE-MolSG.

### Data layout
```
<TARGET>.tar.gz extracts to:
  <TARGET>/ESP_Npy/{ligand,decoy}_<ID>.npy         # GE-MolSG
  <TARGET>/ESP_Npy_MMFF94/{ligand,decoy}_<ID>.npy  # MMFF94
```

### Codebooks
```
experiments/codebooks/ge_molsg_cb.npy   # GE-MolSG geo codebook
experiments/codebooks/mmff94_cb.npy     # MMFF94 codebook
```

### Queries
```
<QUERIES_DIR>/<target>.csv  with a 'query' (or 'queries') column of active IDs
```

## 1 · Configuration
Edit the paths.

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import logging
import sys
import tarfile
from pathlib import Path

import numpy as np
import pandas as pd

# ── Experiment paths (EDIT THESE) ────────────────────────────────────────────
TARGETS = [
    "AA2AR", "ABL1", "ACES", "ADA", "ADRB2", "AMPC", "ANDR", "CSF1R",
    "CXCR4", "DEF", "DRD4", "EGFR", "FA10", "FA7", "FABP4", "FGFR1",
    "FKB1A", "GLCM", "HDAC8", "HIVPR", "HMDH", "HS90A", "ITAL", "KIT",
    "KITH", "LCK", "MAPK2", "MK01", "MT1", "NRAM", "PARP1", "PLK1",
    "PPARA", "PTN1", "PUR2", "RENI", "ROCK1", "SRC", "THRB", "TRY1",
    "TRYB1", "UROK", "XIAP",
]  # comment out any targets you don't want to run
# Data: per-target archives <TARGET>.tar.gz are hosted in one Zenodo record.
# Set ZENODO_BASE_URL and each listed target is downloaded + extracted on demand
# (only the targets in TARGETS are fetched). Each <TARGET>.tar.gz extracts to
# <TARGET>/{ESP_Npy, ESP_Npy_MMFF94, PDB_Files}/ — both arms read from there.
ZENODO_BASE_URL = ""   # e.g. "https://zenodo.org/records/XXXXXXX/files"
DATA_ROOT    = Path("dude_z_data")   # local cache for downloaded/extracted targets
QUERIES_DIR  = Path("experiments/queries")
OUT_DIR      = Path("experiments_out/exp4_gemolsg_vs_mmff94")
CODEBOOK_DIR = Path("experiments/codebooks")

GE_CODEBOOK     = CODEBOOK_DIR / "ge_molsg_cb.npy"  # GE-MolSG geo codebook
MMFF94_CODEBOOK = CODEBOOK_DIR / "mmff94_cb.npy"    # MMFF94 codebook

METHODS = ["GE-MolSG", "MMFF94"]
METRICS = ["EF1%", "BEDROC"]

# ── Descriptor params (shared by both arms) ──────────────────────────────────
K        = 100      # LBO eigenvalues (n_components)
EVALS    = 100      # WKS evals
VAR      = 15       # WKS variance
KNN      = 100      # graph kNN
EW       = 0.3      # electrostatic weight
LAP_NORM = "normalized"
BOF_KNN  = 3

# MMFF94 reads precomputed surfaces (vertices + ESP) from this sub-directory.
MMFF94_PRECOMP_SUBDIR = "ESP_Npy_MMFF94"

FORCE        = False     # recompute even if a target CSV is cached


In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT = OUT_DIR / "results"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout),
              logging.FileHandler(OUT_DIR / "exp4_gemolsg_vs_mmff94.log")],
    force=True,
)
log = logging.getLogger("exp4")
log.info("Experiment 4 — GE-MolSG vs MMFF94")
log.info("Targets: %s", ", ".join(TARGETS))


## 2 · Data / query helpers

In [ ]:
import shutil
import subprocess
import urllib.request


def fetch_target(target, data_root):
    """Ensure <data_root>/<TARGET>/ exists, downloading from Zenodo if needed.

    Looks for an already-extracted ``<TARGET>/ESP_Npy`` first. If absent and
    ``ZENODO_BASE_URL`` is set, downloads ``<ZENODO_BASE_URL>/<TARGET>.tar.gz``
    and extracts it under ``data_root``. Each archive extracts to
    ``<TARGET>/{ESP_Npy, ESP_Npy_MMFF94, PDB_Files}/``.

    Returns the ``<TARGET>/`` directory.
    """
    data_root = Path(data_root)
    for cand in (data_root / target, data_root / target.upper(),
                 data_root / "DUDE-Z" / target):
        if (cand / "ESP_Npy").is_dir():
            return cand

    local_tar = None
    for tar in (data_root / f"{target}.tar.gz",
                data_root / "DUDE-Z" / f"{target}.tar.gz"):
        if tar.exists():
            local_tar = tar
            break

    if local_tar is None:
        if not ZENODO_BASE_URL:
            raise FileNotFoundError(
                f"No local data for '{target}' and ZENODO_BASE_URL is not set.")
        data_root.mkdir(parents=True, exist_ok=True)
        local_tar = data_root / f"{target}.tar.gz"
        url = f"{ZENODO_BASE_URL.rstrip('/')}/{target}.tar.gz"
        log.info("Downloading %s", url)
        try:
            subprocess.run(["curl", "-fSL", "-o", str(local_tar), url], check=True)
        except (FileNotFoundError, subprocess.CalledProcessError):
            urllib.request.urlretrieve(url, local_tar)

    log.info("Extracting %s", local_tar)
    with tarfile.open(local_tar, "r:gz") as tf:
        tf.extractall(data_root)

    for cand in (data_root / target, data_root / target.upper()):
        if (cand / "ESP_Npy").is_dir():
            return cand
    hits = list(data_root.rglob(f"{target}/ESP_Npy")) or list(data_root.rglob("ESP_Npy"))
    if hits:
        return hits[0].parent
    raise FileNotFoundError(f"Extracted '{target}' but found no ESP_Npy under {data_root}")


def resolve_target_dir(target, data_root, work=None):
    """Return the <TARGET>/ directory holding ESP_Npy, ESP_Npy_MMFF94, PDB_Files."""
    return fetch_target(target, data_root)

def resolve_queries_csv(queries_dir, target):
    for name in (f"{target}.csv", f"{target.lower()}.csv", f"{target.upper()}.csv"):
        if (queries_dir / name).exists():
            return queries_dir / name
    raise FileNotFoundError(f"No queries CSV for '{target}' under {queries_dir}")

def load_query_ids(csv_path):
    """Read query IDs from the CSV, using whichever query column is present."""
    df = pd.read_csv(csv_path)
    col = next((c for c in ("queries", "query") if c in df.columns), None)
    if col is None:
        col = df.columns[0] if df.shape[1] == 1 else None
    if col is None:
        raise ValueError(f"{csv_path}: no 'query'/'queries' column found")
    return [str(q).strip() for q in df[col].dropna()]

def list_surface_fns(surf_dir):
    return sorted([f for f in os.listdir(surf_dir) if f.endswith(".npy")], reverse=True)

## 3 · Retrieval metrics
Both methods use chi-squared-kernel similarity. EF1% and BEDROC (α=20).

In [ ]:
from sklearn.metrics.pairwise import chi2_kernel
from rdkit.ML.Scoring import Scoring

def _metrics_from_sim(sim, labels_rest):
    order = np.argsort(sim)[::-1]
    scores = np.column_stack([np.asarray(sim)[order], np.asarray(labels_rest)[order]])
    ef = Scoring.CalcEnrichment(scores, 1, [0.01])
    bedroc = Scoring.CalcBEDROC(scores, 1, 20)
    return {"EF1%": float(ef[0]), "BEDROC": float(bedroc)}

def retrieve_simfn(descs, fns, query_ids, sim_fn):
    """Generic retrieval: sim_fn(descs, i) -> similarities of i vs all others (i removed)."""
    labels = np.asarray([1 if f[0] == "l" else 0 for f in fns])
    stems = [f[:-4] if f.endswith(".npy") else f.rsplit(".", 1)[0] for f in fns]
    rows = []
    for qid in query_ids:
        hits = [j for j, s in enumerate(stems) if s == qid or s.endswith(qid)]
        if not hits:
            log.warning("  query '%s' not found; skipping", qid)
            continue
        i = hits[0]
        rest = [labels[z] for z in range(len(fns)) if z != i]
        sim = sim_fn(descs, i)
        m = _metrics_from_sim(sim, rest)
        m["RefMol"] = stems[i]
        rows.append(m)
    return pd.DataFrame(rows, columns=["RefMol", "EF1%", "BEDROC"])

## 4 · Encoders

Both methods run the same GE-MolSG pipeline (WKS → hard k-NN BoF → chi²),
differing only in the surface source: GE-MolSG reads the DUDE-Z `ESP_Npy`
directories, MMFF94 reads the precomputed `ESP_Npy_MMFF94` sub-directory.

In [ ]:
import ge_molsg as gm

# ── GE-MolSG ─────────────────────────────────────────────────────────────────
def encode_gemolsg(esp_dir, fns):
    """GE-MolSG WKS -> hard k-NN BoF over the geo codebook."""
    surfaces = [gm.load_surface_npy(str(esp_dir / f), name=f[:-4]) for f in fns]
    codebook = np.load(GE_CODEBOOK, allow_pickle=True)
    vecs = []
    for surf in surfaces:
        pts = surf.augmented_points(elec_weight=EW)
        W = gm.compute_affinity(
            pts, n_neighbors=KNN, backend="ckdtree",
            adaptive_bw=True, square_distances=False,
            median_k=100,
        )
        L = gm.graph_laplacian(W, laplacian_type=LAP_NORM)
        ev, evec = gm.compute_eigenpairs(
            L, n_components=K, drop_first=True,
            normalize_vectors=False, eigensolver="arpack",
        )
        wks = gm.wks([ev, evec], evals=EVALS, variance=VAR, l2=True)
        wks = np.nan_to_num(wks)
        vecs.append(gm.knn_histogram(wks, codebook, knn=BOF_KNN))
    V = np.asarray(vecs)
    S = chi2_kernel(V)

    def sim_fn(_descs, i):
        keep = np.arange(S.shape[0]) != i
        return S[i][keep]
    return V, sim_fn, fns

# ── MMFF94 (precomputed) ─────────────────────────────────────────────────────
def encode_mmff94(precomp_dir, fns):
    """MMFF94 precomputed ESP -> WKS -> hard k-NN BoF.

    Same GE-MolSG pipeline as ``encode_gemolsg``; vertices and ESP are read from
    the precomputed ESP_Npy_MMFF94 surfaces instead of the DUDE-Z ESP_Npy ones.
    """
    codebook = np.load(MMFF94_CODEBOOK, allow_pickle=True)
    vecs = []
    for f in fns:
        pre_mol  = np.load(str(precomp_dir / f), allow_pickle=True)
        vertices = pre_mol[0]
        esp      = pre_mol[2]
        graph_features = np.concatenate(
            [vertices, (esp * EW).reshape(-1, 1)], axis=1
        )
        W = gm.compute_affinity(
            graph_features, n_neighbors=KNN, backend="ckdtree",
            adaptive_bw=True, square_distances=False,
            median_k=100,
        )
        L = gm.graph_laplacian(W, laplacian_type=LAP_NORM)
        ev, evec = gm.compute_eigenpairs(
            L, n_components=K, drop_first=True,
            normalize_vectors=False, eigensolver="arpack",
        )
        wks = gm.wks([ev, evec], evals=EVALS, variance=VAR, l2=True)
        wks = np.nan_to_num(wks)
        vecs.append(gm.knn_histogram(wks, codebook, knn=BOF_KNN))
    V = np.asarray(vecs)
    S = chi2_kernel(V)

    def sim_fn(_descs, i):
        keep = np.arange(S.shape[0]) != i
        return S[i][keep]
    return V, sim_fn, fns

## 5 · Per-target driver (iterative, cached)

In [ ]:
def run_target(target):
    out_by_method = {}
    esp_dir = precomp_dir = target_root = base_fns = query_ids = None
    for method in METHODS:
        csv_path = RESULTS_ROOT / method / f"{target}.csv"
        if csv_path.exists() and not FORCE:
            log.info("[%s | %s] cached -> %s", target, method, csv_path)
            out_by_method[method] = pd.read_csv(csv_path)
            continue
        # Lazy-load surface list and query IDs on first non-cached method.
        if esp_dir is None:
            target_root = resolve_target_dir(target, DATA_ROOT, OUT_DIR / "_work")
            esp_dir     = target_root / "ESP_Npy"
            precomp_dir = target_root / MMFF94_PRECOMP_SUBDIR
            base_fns    = list_surface_fns(esp_dir)
            query_ids   = load_query_ids(resolve_queries_csv(QUERIES_DIR, target))
            n_lig = sum(f[0] == "l" for f in base_fns)
            log.info("[%s] %d surfaces (%d ligands, %d decoys), %d queries",
                     target, len(base_fns), n_lig, len(base_fns) - n_lig, len(query_ids))
        log.info("[%s | %s] encoding %d surfaces ...", target, method, len(base_fns))
        try:
            if method == "GE-MolSG":
                descs, sim_fn, fns = encode_gemolsg(esp_dir, base_fns)
            else:  # MMFF94
                descs, sim_fn, fns = encode_mmff94(precomp_dir, base_fns)
        except Exception as e:
            log.error("[%s | %s] encoding failed: %r", target, method, e)
            out_by_method[method] = pd.DataFrame(columns=["RefMol", "EF1%", "BEDROC"])
            continue
        log.info("[%s | %s] retrieval (chi2) ...", target, method)
        df = retrieve_simfn(descs, fns, query_ids, sim_fn)
        csv_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(csv_path, index=False)
        log.info("[%s | %s] %d queries  mean EF1%%=%.3f  mean BEDROC=%.3f -> %s",
                 target, method, len(df),
                 df["EF1%"].mean() if len(df) else float("nan"),
                 df["BEDROC"].mean() if len(df) else float("nan"), csv_path)
        out_by_method[method] = df
    return out_by_method

sampled_data = {m: {} for m in METHODS}
for ti, target in enumerate(TARGETS, 1):
    log.info("==== target %d/%d : %s ====", ti, len(TARGETS), target)
    try:
        by_method = run_target(target)
    except FileNotFoundError as e:
        log.error("skipping %s: %s", target, e)
        continue
    for method, df in by_method.items():
        if len(df):
            sampled_data[method][target] = df

log.info("encoding + retrieval complete")

## 6 · Summary

In [ ]:
rows = []
for method in sampled_data:
    for t in TARGETS:
        if t in sampled_data[method] and len(sampled_data[method][t]):
            df = sampled_data[method][t]
            rows.append(dict(method=method, target=t,
                             mean_EF1=df["EF1%"].mean(),
                             mean_BEDROC=df["BEDROC"].mean(),
                             n_queries=len(df)))
summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "summary_per_target.csv", index=False)
log.info("wrote summary_per_target.csv (%d rows)", len(summary))
summary

## 7 · Scatter / line plots — BEDROC and EF1%
Per-target mean over the target's queries, for GE-MolSG and MMFF94.

In [ ]:
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", font_scale=1.0)
except Exception:
    pass

COLOUR_MAP     = {"GE-MolSG": "#0072B2", "MMFF94": "#E69F00"}
METHOD_MARKERS = {"GE-MolSG": "s",        "MMFF94": "D"}
SCATTER_METHODS = ["GE-MolSG", "MMFF94"]

def scatter(metric):
    tsorted = sorted(TARGETS)
    x_pos = np.arange(len(tsorted))
    y_min = 0.0
    y_max = 50.0 if metric == "EF1%" else 0.9

    fig_width = max(8, len(tsorted) * 0.35)
    fig, ax = plt.subplots(figsize=(fig_width, 4.5), facecolor="white")
    ax.set_facecolor("#EBEBEB")
    ax.grid(axis="y", linestyle="-", color="white", linewidth=0.8, zorder=0)
    ax.grid(axis="x", linestyle="-", color="white", linewidth=0.8, zorder=0)
    ax.margins(x=0.02)

    for method in SCATTER_METHODS:
        means = [sampled_data[method][t][metric].mean()
                 if t in sampled_data.get(method, {}) and len(sampled_data[method][t])
                 else np.nan
                 for t in tsorted]
        col = COLOUR_MAP.get(method, "#888888")
        ax.plot(x_pos, means, color=col, linewidth=1.4,
                marker=METHOD_MARKERS.get(method, "o"), markersize=6,
                markerfacecolor="white", markeredgecolor=col, markeredgewidth=1.5,
                label=method, zorder=3)

    ax.set_xlim(-0.5, len(tsorted) - 0.5)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(tsorted, rotation=90, ha="center", fontsize=9)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xlabel("Target", fontsize=12)
    title = {"BEDROC": "DUDE-Z Mean BEDROC (α=20) Retrieval Performance",
             "EF1%":   "DUDE-Z Mean EF1% Retrieval Performance"}.get(
                 metric, f"DUDE-Z Mean {metric} Retrieval Performance")
    ax.set_title(title, fontsize=14, pad=6)
    ax.legend(fontsize=10, bbox_to_anchor=(1.01, 1), loc="upper left",
              borderaxespad=0.0, framealpha=0.85)
    plt.tight_layout()
    path = OUT_DIR / f"scatter_{metric.replace('%','pct')}.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    log.info("saved %s", path)

scatter("BEDROC")
scatter("EF1%")

## 8 · Wilcoxon signed-rank test — focused on GE-MolSG
Paired on per-target means; GE-MolSG vs MMFF94, per metric. p-values unadjusted.

In [ ]:
from scipy.stats import wilcoxon

WILCOXON_METHOD = "GE-MolSG"
alpha = 0.05
records = []

for metric in METRICS:
    for baseline in sampled_data:
        if baseline == WILCOXON_METHOD:
            continue
        paired_focal, paired_base = [], []
        for target in TARGETS:
            if (target in sampled_data[WILCOXON_METHOD]
                    and len(sampled_data[WILCOXON_METHOD].get(target, []))
                    and target in sampled_data[baseline]
                    and len(sampled_data[baseline].get(target, []))):
                paired_focal.append(sampled_data[WILCOXON_METHOD][target][metric].mean())
                paired_base.append(sampled_data[baseline][target][metric].mean())
        n_pairs = len(paired_focal)
        if n_pairs < 4:
            records.append(dict(Metric=metric, Baseline=baseline, N_targets=n_pairs,
                                W=np.nan, p_value=np.nan, Significant="—",
                                Direction="insufficient data"))
            continue
        diffs = np.array(paired_focal) - np.array(paired_base)
        if np.all(diffs == 0):
            records.append(dict(Metric=metric, Baseline=baseline, N_targets=n_pairs,
                                W=np.nan, p_value=np.nan, Significant="—",
                                Direction="identical"))
            continue
        stat, p = wilcoxon(paired_focal, paired_base, alternative="two-sided")
        records.append(dict(Metric=metric, Baseline=baseline, N_targets=n_pairs,
                            W=round(stat, 3), p_value=round(p, 4),
                            Significant="✓" if p < alpha else "✗",
                            Direction="better" if np.mean(diffs) > 0 else "worse"))

sig_df = pd.DataFrame(records)
if len(sig_df):
    sig_df = sig_df.sort_values(["Metric", "p_value"])
sig_df.to_csv(
    OUT_DIR / f"wilcoxon_{WILCOXON_METHOD.replace(' ','_')}.csv",
    index=False)

print(f"Wilcoxon signed-rank test: {WILCOXON_METHOD} vs all others")
print(f"Paired on per-target means; α = {alpha}\n")
for metric in METRICS:
    if "Metric" not in sig_df.columns:
        break
    sub = sig_df[sig_df.Metric == metric][
        ["Baseline", "N_targets", "W", "p_value", "Significant", "Direction"]]
    print(f"── {metric} {'─'*40}")
    print(sub.to_string(index=False))
    print()
sig_df